In [214]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import ast

In [298]:
df = pd.read_csv('movies.csv') 
df = df[['title', 'budget', 'revenue', 'genres', 'belongs_to_collection', 
          'release_date', 'vote_average', 'vote_count', 'runtime']]
df = df[(df['budget']>0) & (df['revenue']>0)] #remove any rows where either budget are 0/unknown 
df.info()

<class 'pandas.DataFrame'>
Index: 4991 entries, 0 to 9994
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  4991 non-null   str    
 1   budget                 4991 non-null   float64
 2   revenue                4991 non-null   int64  
 3   genres                 4991 non-null   str    
 4   belongs_to_collection  1632 non-null   str    
 5   release_date           4991 non-null   str    
 6   vote_average           4991 non-null   float64
 7   vote_count             4991 non-null   int64  
 8   runtime                4991 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 389.9 KB


In [299]:
#clean up genres, extract just name

def extract_genres(genres_str):
    genres = ast.literal_eval(genres_str) #convert to dictionary
    return [g['name'] for g in genres]

df['genres'] = df['genres'].apply(extract_genres)


In [300]:
#extract name only from collection
def extract_collection(collection_str):
    if pd.isna(collection_str):
        return None
    else:
        collection_dict = ast.literal_eval(collection_str) #convert to dictionary
        return collection_dict['name']
df['collection'] = df['belongs_to_collection'].apply(extract_collection)



In [301]:

#belongs_to_collection true/false 
df['belongs_to_collection'] = df['belongs_to_collection'].notna()
df = df.rename(columns={'belongs_to_collection': 'is_franchise'})


In [302]:
#return on investment for each movie
df['roi'] = (df['revenue'] - df['budget'])/df['budget']
df.head(20)

,title,budget,revenue,genres,is_franchise,release_date,vote_average,vote_count,runtime,collection,roi
0,Karate Kid: Legends,45000000.0,104560790,"[Action, Adventure, Drama]",True,2025-05-08,7.281,367,94,The Karate Kid Collection,1.323573
1,Ballerina,90000000.0,131611905,"[Action, Thriller, Crime]",True,2025-06-04,7.453,944,125,Ballerina Collection,0.462355
2,Superman,225000000.0,217000000,"[Science Fiction, Adventure, Action]",False,2025-07-09,7.470,538,130,NaN,-0.035556
7,Jurassic World Rebirth,180000000.0,529463000,"[Science Fiction, Adventure, Action]",True,2025-07-01,6.400,566,134,Jurassic Park Collection,1.941461
8,Thunderbolts*,180000000.0,382027956,"[Action, Science Fiction, Adventure]",False,2025-04-30,7.428,1767,127,NaN,1.122378
10,Final Destination Bloodlines,50000000.0,285153000,"[Horror, Mystery]",True,2025-05-14,7.200,1605,110,Final Destination Collection,4.703060
12,Lilo & Stitch,100000000.0,994264677,"[Family, Science Fiction, Comedy, Adventure]",True,2025-05-17,7.154,827,108,Lilo & Stitch (Live-Action) Collection,8.942647
14,How to Train Your Dragon,150000000.0,560773000,"[Fantasy, Family, Action]",True,2025-06-06,7.887,630,125,How to Train Your Dragon (Live-Action) Collection,2.738487
15,Bring Her Back,15000000.0,22878745,[Horror],False,2025-05-28,7.425,242,104,NaN,0.525250
16,F1,200000000.0,393395000,"[Action, Drama]",False,2025-06-25,7.664,727,156,NaN,0.966975


In [303]:
#remove duplicates, many are remakes so drop exact duplicates only (same release date)
df['title'].duplicated().sum() #172 duplicates
df = df.drop_duplicates(subset=['title', 'release_date']) 
df['release_date'] = pd.to_datetime(df['release_date']) 

In [304]:
df_genres = df.explode('genres')
df_genres.head(10)

,title,budget,revenue,genres,is_franchise,release_date,vote_average,vote_count,runtime,collection,roi
0,Karate Kid: Legends,45000000.0,104560790,Action,True,2025-05-08,7.281,367,94,The Karate Kid Collection,1.323573
0,Karate Kid: Legends,45000000.0,104560790,Adventure,True,2025-05-08,7.281,367,94,The Karate Kid Collection,1.323573
0,Karate Kid: Legends,45000000.0,104560790,Drama,True,2025-05-08,7.281,367,94,The Karate Kid Collection,1.323573
1,Ballerina,90000000.0,131611905,Action,True,2025-06-04,7.453,944,125,Ballerina Collection,0.462355
1,Ballerina,90000000.0,131611905,Thriller,True,2025-06-04,7.453,944,125,Ballerina Collection,0.462355
1,Ballerina,90000000.0,131611905,Crime,True,2025-06-04,7.453,944,125,Ballerina Collection,0.462355
2,Superman,225000000.0,217000000,Science Fiction,False,2025-07-09,7.470,538,130,NaN,-0.035556
2,Superman,225000000.0,217000000,Adventure,False,2025-07-09,7.470,538,130,NaN,-0.035556
2,Superman,225000000.0,217000000,Action,False,2025-07-09,7.470,538,130,NaN,-0.035556
7,Jurassic World Rebirth,180000000.0,529463000,Science Fiction,True,2025-07-01,6.400,566,134,Jurassic Park Collection,1.941461


In [305]:
engine = create_engine('postgresql://localhost/movies_db')
df.to_sql('movies', con=engine, if_exists='replace', index=False)
df_genres.to_sql(name='movie_genres', con=engine, if_exists='replace', index=False)

662

In [223]:
df.info()

<class 'pandas.DataFrame'>
Index: 4953 entries, 0 to 9994
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   title         4953 non-null   str           
 1   budget        4953 non-null   float64       
 2   revenue       4953 non-null   int64         
 3   genres        4953 non-null   object        
 4   is_franchise  4953 non-null   bool          
 5   release_date  4953 non-null   datetime64[us]
 6   vote_average  4953 non-null   float64       
 7   vote_count    4953 non-null   int64         
 8   runtime       4953 non-null   int64         
 9   roi           4953 non-null   float64       
dtypes: bool(1), datetime64[us](1), float64(3), int64(3), object(1), str(1)
memory usage: 391.8+ KB
